# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv")

df.columns = df.columns.str.lower().str.replace(" ", "_")

df.head()

,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


In [24]:
df['effective_to_date'] = pd.to_datetime(
    df['effective_to_date'],
    format='%d/%m/%y',
    errors='coerce'
)

df['month'] = df['effective_to_date'].dt.month_name()

In [14]:
low_claim_yes = df[(df['total_claim_amount'] < 1000) & (df['response'].str.lower() == 'yes')].copy()
low_claim_yes.shape
low_claim_yes.head()

,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,January
8,8,FM55990,California,5989.773931,Yes,Premium,College,2011-01-19,Employed,M,...,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN,January
15,15,CW49887,California,4626.801093,Yes,Basic,Master,2011-01-16,Employed,F,...,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN,January
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2011-02-26,Employed,F,...,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A,February
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2011-02-28,Employed,F,...,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN,February


In [16]:
responders = df[df['response'].str.lower() == 'yes'].copy()

pivot_avg = pd.pivot_table(
    responders,
    values=['monthly_premium_auto', 'customer_lifetime_value'],
    index=['policy_type', 'gender'],
    aggfunc='mean'
).round(2)

pivot_avg

customer_lifetime_value  monthly_premium_auto
policy_type    gender                                               
Corporate Auto F                       7712.63                 94.30
               M                       7944.47                 92.19
Personal Auto  F                       8339.79                 99.00
               M                       7448.38                 91.09
Special Auto   F                       7691.58                 92.31
               M                       8247.09                 86.34

In [18]:
comparison = (
    responders
    .groupby(['policy_type', 'gender'])
    .agg(
        avg_monthly_premium = ('monthly_premium_auto', 'mean'),
        avg_clv = ('customer_lifetime_value', 'mean'),
        avg_total_claim = ('total_claim_amount', 'mean'),
        median_total_claim = ('total_claim_amount', 'median'),
        count = ('customer', 'count')
    )
    .round(2)
    .reset_index()
)

comparison

,policy_type,gender,avg_monthly_premium,avg_clv,avg_total_claim,median_total_claim,count
0,Corporate Auto,F,94.30,7712.63,433.74,420.04,169
1,Corporate Auto,M,92.19,7944.47,408.58,364.80,154
2,Personal Auto,F,99.00,8339.79,452.97,424.33,540
3,Personal Auto,M,91.09,7448.38,457.01,412.80,536
4,Special Auto,F,92.31,7691.58,453.28,420.04,35
5,Special Auto,M,86.34,8247.09,429.53,345.60,32


In [19]:
customers_by_state = (
    df.groupby('state')['customer'].nunique()
    .reset_index(name='n_customers')
    .sort_values('n_customers', ascending=False)
)

customers_by_state_top = customers_by_state[customers_by_state['n_customers'] > 500]

customers_by_state, customers_by_state_top

(        state  n_customers
 1  California         3150
 3      Oregon         2601
 0     Arizona         1703
 2      Nevada          882
 4  Washington          798,
         state  n_customers
 1  California         3150
 3      Oregon         2601
 0     Arizona         1703
 2      Nevada          882
 4  Washington          798)

In [20]:
clv_stats = (
    df.groupby(['education', 'gender'])['customer_lifetime_value']
    .agg(['max', 'min', 'median'])
    .round(2)
    .reset_index()
)

clv_stats

,education,gender,max,min,median
0,Bachelor,F,73225.96,1904.00,5640.51
1,Bachelor,M,67907.27,1898.01,5548.03
2,College,F,61850.19,1898.68,5623.61
3,College,M,61134.68,1918.12,6005.85
4,Doctor,F,44856.11,2395.57,5332.46
5,Doctor,M,32677.34,2267.60,5577.67
6,High School or Below,F,55277.45,2144.92,6039.55
7,High School or Below,M,83325.38,1940.98,6286.73
8,Master,F,51016.07,2417.78,5729.86
9,Master,M,50568.26,2272.31,5579.10


In [21]:
policies_state_month = (
    df.groupby(['state', 'month'])['policy']
    .count()
    .reset_index(name='n_policies')
)

policies_pivot = policies_state_month.pivot_table(
    index='state', columns='month', values='n_policies', fill_value=0
)

month_order = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]
policies_pivot = policies_pivot.reindex(columns=[m for m in month_order if m in policies_pivot.columns])

policies_pivot

month,January,February
state,,
Arizona,1008.0,929.0
California,1918.0,1634.0
Nevada,551.0,442.0
Oregon,1565.0,1344.0
Washington,463.0,425.0


In [22]:
state_totals = (
    policies_state_month.groupby('state')['n_policies']
    .sum()
    .reset_index()
    .sort_values('n_policies', ascending=False)
)

top3_states = state_totals.head(3)['state'].tolist()

top3_pivot = policies_state_month[policies_state_month['state'].isin(top3_states)].pivot_table(
    index='state', columns='month', values='n_policies', fill_value=0
)

top3_pivot = top3_pivot.reindex(columns=[m for m in month_order if m in top3_pivot.columns])

top3_pivot

month,January,February
state,,
Arizona,1008.0,929.0
California,1918.0,1634.0
Oregon,1565.0,1344.0


In [23]:
channel_response = (
    df.groupby('sales_channel')['response']
    .apply(lambda s: (s.str.lower() == 'yes').sum() / s.count())
    .reset_index(name='response_rate')
)

channel_response['response_rate'] = (channel_response['response_rate']*100).round(2)
channel_response.sort_values('response_rate', ascending=False)

,sales_channel,response_rate
0,Agent,19.07
3,Web,11.71
1,Branch,11.38
2,Call Center,10.98
